In [1]:
import sys
sys.path.append("../")

from plotting.simulation_plotters import *
from plotting.bads_plotters import *
from experiments.simulation_experiments import *
from generators.rsc_z_generator import *
from helpers.helpers import *
import numpy as np

#This sets the seed for random sampling to be constant such that randomly sampling from an identical distribution twice will always produce identical samples.
np.random.seed(seed=231321312)

%reload_ext autoreload
%aimport plotting.simulation_plotters
%aimport helpers.helpers
%aimport generators.rsc_z_generator
%aimport experiments.simulation_experiments
%autoreload 2

In [5]:
## Setup Parameters ## 
d = [3, 5, 7, 9, 11, 13, 15, 17]
use_d_for_rounds = False
ler_target = 0.005

## Defect Cases (Case 2 & Case 4) ##
p_defect_values = [0.0, 0.05, 0.25, 0.50, 0.75]
defect_type = ['center data', 'edge measure']

## Overview ##

This tools centers around the creation of "experiments" to facilitate rapid experimentation and visualization of performance under a variety of noise configurations. All experiments use the same Stim generator to generate the underlying circuit, but vary in the setup of the input parameters and data parsing to more readily break the sample data into usable DataFrames by the plotters. The four main experiments that are currently pre-created are as follows:
* Uniform Homogeneous Noise (`uniform_homogeneous_noise_model_simulation_distance`): Uses a single noise value, `p`, for all qubits in the lattice. Returns a single DF containing all distances provided.
* Homogeneous Noise (`homogeneous_noise_model_simulation`): Same as uniform with the exception that individual defects can be manually specified. Returns a list of DF for each distance given in the input.
* Heterogeneous Noise Model (`heterogeneous_noise_model_simulation`): Given an array of distances and standard deviations as input, samples lattices that have normally distributed noise with each combination of mean and standard deviation for each distance provided. This is useful to see how a variety of noise deviation impacts the LER as the overall lattice becomes noiser (i.e. as the mean noise of all qubits increases.) Returns a list of DF for each distance given in the input.
* Heterogeneous Noise Model (`heterogeneous_noise_model_with_defect_simulation`) Same as above except with defect locations and a range of physical error rates for the specified defects for a single standard deviation. This is useful to see how, for a single deviation value, how various degrees of defect qubit physical error rates impact the LER as the overall lattice becomes noiser (i.e. as the mean noise of all non-defective qubits increases.) Returns a list of DF for each distance given in the input.

The plotting element has pre-built configurations (axis titles, legend titles, and plot titles) for each of the above cases to enable rapid experimentation. However, the plotting function allows for customization of both the x and y axis to create experimental cases that are different than the above.

## Case 1: Uniform Homogeneous Noise ##

In [ ]:
## Run Simulation w/ Parameters ## 
homogeneous_uniform_stat = uniform_homogeneous_noise_model_simulation_distance([3,5,7,9,11,13,15,17], p_values=[round_to_sig_figs(i, 15) for i in np.logspace(-3, -1, 20).tolist()]) 

In [ ]:
## Plot simulation data ## 
plot_simulation_data(homogeneous_uniform_stat, ler_target=0.005, save_figure_path=f'case1/homogeneous-d{d[0]}-{d[7]}') 

In [ ]:
## The plots can be saved by specifying the path to save the figures to. This will, by default, always save to the /data subfolder in the directory
plot_simulation_data(homogeneous_uniform_stat, save_figure_path=f'case1/homogeneous-d{d[0]}-{d[7]}')

In [ ]:
## Each simulation stats DF can be saved as a CSV in the \data folder under an existing or new experiment folder ##
# This function saves the underlying Pandas DF to a CSV in a subdirectory of the repository, /data. Within this data folder, you can specify additional subdirectories via the 'subfolder' parameter
stats_df_to_csv(df=homogeneous_uniform_stat, filename='case1-uniform-homogeneous-d3-17', subfolder='case1')

In [ ]:
## For each plot, the intersections with the ler target target can be shown and/or offloaded into a DataFrame for further use
# The intersections (referred to as the "Boundaries of Acceptable Defectiveness" or BADs) can also be saved as a cvs, similar to the data used to generate a specific plot with the following:
stats_df_to_csv(generate_BADs(homogeneous_uniform_stat, ler_target=ler_target), 'case1-BADs', 'case1')

## Case 2: Homogeneous Noise With Center Defect ##

In [ ]:
## Setup Parameters ##
d = [3, 5, 7, 9, 11, 13, 15, 17]
p_defect_values = [0.0, 0.05, 0.25, 0.50, 0.75]

In [ ]:
## Run Simulation w/ Parameters (Estimated Runtime (Rounds = 3): ~ 6 Minutes; (Rounds = distance): ~26 Minutes) ## 
homogeneous_model_stat = [homogeneous_noise_model_simulation(d[:4], p_defect_values=p_defect_values, use_d_for_rounds=use_d_for_rounds, defect_type=defect_type),
                          homogeneous_noise_model_simulation(d[4:], p_defect_values=p_defect_values, use_d_for_rounds=use_d_for_rounds, defect_type=defect_type)]


In [ ]:
## Plot simulation data ##
plot_multi_distance_simulation_data(homogeneous_model_stat[0], ler_target=ler_target)
plot_multi_distance_simulation_data(homogeneous_model_stat[1], ler_target=ler_target)

In [ ]:
# Save all data for Case 2
case2_results = [*homogeneous_model_stat[0], *homogeneous_model_stat[1]]
## Save plots
plot_multi_distance_simulation_data(homogeneous_model_stat[0], save_figure_path=f'case2/homogeneous-defect-d{d[0]}-{d[3]}', ler_target=ler_target)
plot_multi_distance_simulation_data(homogeneous_model_stat[1], save_figure_path=f'case2/homogeneous-defect-d{d[4]}-{d[7]}', ler_target=ler_target)

## Save simulation results ##
stats_df_to_csv(df=case2_results, filename='case2-uniform-homogeneous-d3-17', subfolder='case2')

## Save BADs data
stats_df_to_csv(generate_BADs(concat_stats_df(case2_results), ler_target=ler_target), 'case2-BADs', 'case2')

## Case 3: Heterogeneous Noise ##

In [ ]:
## Setup Parameters ##
alpha_scalar = [0.0, 0.01, 0.3, 0.6, 0.9]
np.random.seed(seed=6275917)

In [ ]:
## Run Simulation w/ Parameters (Estimated Runtime (Rounds = 3): ~ 5 Minutes; (Rounds = d): ~28 Minutes) ## 
heterogeneous_model_stat = [heterogeneous_noise_model_simulation(d[:4], p_sigma_values=alpha_scalar, use_d_for_rounds=use_d_for_rounds, use_p_sigma_values_as_scalars=True),
                            heterogeneous_noise_model_simulation(d[4:], p_sigma_values=alpha_scalar, use_d_for_rounds=use_d_for_rounds, use_p_sigma_values_as_scalars=True)]

In [ ]:
## Plot simulation data ##
plot_multi_distance_simulation_data(heterogeneous_model_stat[0], x_range=[10**(-3), 10**(-1)], ler_target=ler_target)
plot_multi_distance_simulation_data(heterogeneous_model_stat[1], x_range=[10**(-3), 10**(-1)], ler_target=ler_target)

In [ ]:
## Save all data for Case 3
case3_results = [*heterogeneous_model_stat[0], *heterogeneous_model_stat[1]]

# Save plots
plot_multi_distance_simulation_data(heterogeneous_model_stat[0], x_range=[10**(-3), 10**(-1)], save_figure_path=f'case3/heterogeneous-d{d[0]}-{d[3]}', ler_target=ler_target)
plot_multi_distance_simulation_data(heterogeneous_model_stat[1], x_range=[10**(-3), 10**(-1)], save_figure_path=f'case3/heterogeneous-d{d[4]}-{d[7]}', ler_target=ler_target)

## Save simulation results ##
stats_df_to_csv(df=case3_results, filename='case3-heterogeneous-d3-17', subfolder='case3')

## Save BADs data
stats_df_to_csv(generate_BADs(concat_stats_df(case3_results), ler_target=ler_target), 'case3-BADs', 'case3')

## Case 4: Heterogeneous Noise With Center Data & Edge Measure Defect ##

In [ ]:
## Setup Parameters ##
alpha_scalar = 0.6
defect_type = ['center data', 'edge measure']
np.random.seed(seed=76541532)

In [ ]:
## Run Simulation w/ Parameters (Estimated Runtime: ~ 5 Minutes) ## 
heterogeneous_defect_model_stat = [heterogeneous_noise_model_with_defect_simulation(d[:4], scalar=alpha_scalar, use_p_sigma_values_as_scalars=True, p_defect_values=p_defect_values, use_d_for_rounds=use_d_for_rounds, defect_type=defect_type),
                                   heterogeneous_noise_model_with_defect_simulation(d[4:], scalar=alpha_scalar, use_p_sigma_values_as_scalars=True, p_defect_values=p_defect_values, use_d_for_rounds=use_d_for_rounds, defect_type=defect_type)]

In [ ]:
## Plot simulation data ##
plot_multi_distance_simulation_data(heterogeneous_defect_model_stat[0], x_range=[10**(-3), 10**(-1)], y_range=[10**-4, 1])
plot_multi_distance_simulation_data(heterogeneous_defect_model_stat[1], x_range=[10**(-3), 10**(-1)], y_range=[10**-4, 1])

In [ ]:
# # Save all data for Case 4
case4_results = [*heterogeneous_defect_model_stat[0], *heterogeneous_defect_model_stat[1]]

## Save plots
plot_multi_distance_simulation_data(heterogeneous_defect_model_stat[0], x_range=[10**(-3), 10**(-1)], save_figure_path=f'case4/heterogeneous-defect-d{d[0]}-{d[3]}', ler_target=ler_target)
plot_multi_distance_simulation_data(heterogeneous_defect_model_stat[1], x_range=[10**(-3), 10**(-1)], save_figure_path=f'case4/heterogeneous-defect-d{d[4]}-{d[7]}', ler_target=ler_target)

## Save simulation results ##
stats_df_to_csv(df=case4_results, filename='case4-heterogeneous-defect-d3-17', subfolder='case4')

## Save BADs data
stats_df_to_csv(generate_BADs(concat_stats_df(case4_results), ler_target=ler_target), 'case4-BADs', 'case4')